# 03 文档解析与分割

> **任务目标**：将第二阶段输出的 `oa_comm_slim.jsonl`（4,557,627 篇）按已验证的分割策略，生成适合向量化检索的文本块数据集。

## 分割策略（来自第二阶段）

| 条件 | 策略 | 参数 |
|------|------|------|
| token ≤ 512 | 不分割（单块） | 约 86% 文献 |
| token > 512 | 滑动窗口 | chunk_size=400, overlap=80 |

## 执行流程

- **C0**: 环境配置
- **C1**: 数据加载（验证输入）
- **C2**: 分割处理（支持断点续传）
- **C3**: 保存结果
- **C4**: 预览结果
- **C5**: 质量验证

---
## 【C0】环境配置

In [1]:
import sys
import os
from pathlib import Path

# 添加 src 到路径
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 配置数据根目录
if os.name == "nt":  # Windows
    DATA_ROOT = Path("E:/med-llm-rag-datasets")
else:  # Mac
    DATA_ROOT = Path("/Volumes/Lexar/med-llm-rag-datasets")

# 路径配置
INPUT_JSONL = DATA_ROOT / "processed" / "oa_comm_slim.jsonl"
OUTPUT_JSONL = DATA_ROOT / "processed" / "oa_comm_chunks.jsonl"  # 全量输出（新文件，不修改原文件）
CHUNK_CONFIG = Path("../../02 数据处理/outputs/tables/chunk_strategy_config.json")

# 验证样本输出路径（保存到工程目录，用于 git 提交）
SAMPLE_OUTPUT = Path("../data/processed/chunks_sample.jsonl")

print(f"数据根目录: {DATA_ROOT}")
print(f"输入文件: {INPUT_JSONL}")
print(f"全量输出: {OUTPUT_JSONL} (新建文件，不修改原slim)")
print(f"验证样本: {SAMPLE_OUTPUT.resolve()} (工程目录，可git提交)")
print(f"策略配置: {CHUNK_CONFIG.resolve()}")

数据根目录: E:\med-llm-rag-datasets
输入文件: E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl
全量输出: E:\med-llm-rag-datasets\processed\oa_comm_chunks.jsonl (新建文件，不修改原slim)
验证样本: D:\谷歌\03 文档解析与分割\data\processed\chunks_sample.jsonl (工程目录，可git提交)
策略配置: D:\谷歌\02 数据处理\outputs\tables\chunk_strategy_config.json


In [2]:
# 验证路径
assert DATA_ROOT.exists(), f"数据根目录不存在: {DATA_ROOT}"
assert INPUT_JSONL.exists(), f"输入文件不存在: {INPUT_JSONL}"
assert CHUNK_CONFIG.exists(), f"策略配置不存在: {CHUNK_CONFIG}"

# 显示输入文件大小
input_size_mb = INPUT_JSONL.stat().st_size / (1024 * 1024)
print(f"✅ 路径验证通过")
print(f"输入文件大小: {input_size_mb:,.1f} MB")

✅ 路径验证通过
输入文件大小: 8,484.5 MB


In [3]:
# 加载分割策略配置
import json

with open(CHUNK_CONFIG, "r", encoding="utf-8") as f:
    config = json.load(f)

print("=== 分割策略配置 ===")
for k, v in config.items():
    print(f"  {k}: {v}")

=== 分割策略配置 ===
  retrieval_unit: title+abstract
  retrieval_token_limit: 512
  retrieval_chunk_size: 400
  retrieval_chunk_overlap: 80
  body_chunk_size: 512
  body_chunk_overlap: 80
  drop_missing_abstract: True
  drop_short_abstract: False
  short_abstract_char_threshold: 50
  notes: ['主体 ~85% title+abstract ≤512：单 chunk', '长尾 ~14% >512：RecursiveCharacterTextSplitter', 'body 一律 sliding_window，与摘要分开', '无 abstract：丢弃并记录 pmcid']


---
## 【C1】数据加载与验证

验证输入数据格式，预览样本。

In [4]:
import pandas as pd

# 读取前 5 行预览
preview_rows = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        preview_rows.append(json.loads(line.strip()))

df_preview = pd.DataFrame(preview_rows)
print(f"预览前 5 行，字段: {list(df_preview.columns)}")
df_preview[["pmcid", "title", "abstract"]].head()

预览前 5 行，字段: ['pmcid', 'pmid', 'title', 'abstract', 'journal', 'pub_year', 'pub_date', 'n_chars_abstract', 'n_chars_body']


,pmcid,title,abstract
0,PMC176545,The Transcriptome of the Intraerythrocytic Dev...,Plasmodium falciparum is the causative agent o...
1,PMC176546,DNA Analysis Indicates That Asian Elephants Ar...,The origin of Borneo's elephants is controvers...
2,PMC193604,Drosophila Free-Running Rhythms Require Interc...,Robust self-sustained oscillations are a ubiqu...
3,PMC193605,From Gene Trees to Organismal Phylogeny in Pro...,The rapid increase in published genomic sequen...
4,PMC212319,The Guanine Nucleotide Exchange Factor ARNO me...,Phospholipase D (PLD) is involved in many sign...


In [5]:
# 统计总行数（可选，大文件耗时）
SKIP_LINE_COUNT = True  # 设为 False 可统计总行数

if not SKIP_LINE_COUNT:
    print("正在统计总行数...")
    total_lines = sum(1 for _ in open(INPUT_JSONL, "r", encoding="utf-8"))
    print(f"输入文件总行数: {total_lines:,}")
else:
    print("跳过行数统计（已知约 4,557,627 行）")

跳过行数统计（已知约 4,557,627 行）


---
## 【C2】实施分割策略

使用 `DocumentChunker` 进行分割，支持断点续传。

### 重要说明

- **不修改原文件**：输出到新文件 `oa_comm_chunks.jsonl`，原始 `oa_comm_slim.jsonl` 保持不变
- **调试模式**：`LIMIT=1000` 时，结果同时保存到工程目录 `data/processed/chunks_sample.jsonl`（可 git 提交）
- **全量模式**：`LIMIT=None` 时，仅输出到外接硬盘

### 配置说明

| 参数 | 说明 |
|------|------|
| `LIMIT` | 限制处理数量；`1000` 调试（输出验证样本）；`None` 全量 |
| `RESUME` | 是否启用断点续传 |
| `SKIP_CHUNKING` | 设为 `True` 跳过分割直接查看结果 |
| `SAVE_SAMPLE` | 是否保存验证样本到工程目录 |

In [6]:
# === 分割配置 ===
LIMIT = 1000        # 调试：先处理 1000 篇；全量设为 None
RESUME = True       # 启用断点续传
SKIP_CHUNKING = False  # 设为 True 跳过分割
SAVE_SAMPLE = True  # 保存验证样本到工程目录（用于 git 提交）

In [7]:
from chunker import DocumentChunker

if not SKIP_CHUNKING:
    # 初始化分割器（使用第二阶段配置）
    chunker = DocumentChunker(
        token_limit=config["retrieval_token_limit"],
        chunk_size=config["retrieval_chunk_size"],
        chunk_overlap=config["retrieval_chunk_overlap"],
    )
    
    print(f"分割器配置:")
    print(f"  - token_limit: {chunker.token_limit}")
    print(f"  - chunk_size: {chunker.chunk_size}")
    print(f"  - chunk_overlap: {chunker.chunk_overlap}")
    print(f"  - tokenizer: {chunker.tokenizer_name}")
else:
    print("[跳过] SKIP_CHUNKING = True")

c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


分割器配置:
  - token_limit: 512
  - chunk_size: 400
  - chunk_overlap: 80
  - tokenizer: sentence-transformers/all-MiniLM-L6-v2


In [8]:
# 执行分割
if not SKIP_CHUNKING:
    print(f"开始分割...")
    print(f"  输入: {INPUT_JSONL}")
    print(f"  输出: {OUTPUT_JSONL}")
    print(f"  限制: {LIMIT if LIMIT else '无（全量处理）'}")
    print(f"  断点续传: {RESUME}")
    print()
    
    stats = chunker.process_jsonl_stream(
        input_path=INPUT_JSONL,
        output_path=OUTPUT_JSONL,
        limit=LIMIT,
        resume=RESUME,
    )
    
    print("\n=== 分割完成 ===")
    for k, v in stats.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v:,}")
    
    # 保存验证样本到工程目录
    if SAVE_SAMPLE and LIMIT is not None:
        import shutil
        SAMPLE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(OUTPUT_JSONL, SAMPLE_OUTPUT)
        print(f"\n✅ 验证样本已保存: {SAMPLE_OUTPUT.resolve()}")
        print(f"   （可通过 git 提交）")
else:
    print("[跳过] SKIP_CHUNKING = True")

开始分割...
  输入: E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl
  输出: E:\med-llm-rag-datasets\processed\oa_comm_chunks.jsonl
  限制: 1000
  断点续传: True



分割文档:   0%|          | 0/1000 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\10138\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.micro


=== 分割完成 ===
  processed_docs: 1,000
  total_chunks: 1,267
  single_count: 888
  multi_count: 112
  single_ratio: 0.8880
  multi_ratio: 0.1120
  avg_chunks_per_doc: 1.2670

✅ 验证样本已保存: D:\谷歌\03 文档解析与分割\data\processed\chunks_sample.jsonl
   （可通过 git 提交）


---
## 【C3】保存结果统计

保存处理配置和统计信息到 `outputs/tables/`。

In [5]:
import pandas as pd
from datetime import datetime

# 从本地验证样本统计（不依赖外接硬盘的进度文件）
if SAMPLE_OUTPUT.exists():
    # 直接从验证样本文件统计
    sample_chunks = []
    with open(SAMPLE_OUTPUT, "r", encoding="utf-8") as f:
        for line in f:
            sample_chunks.append(json.loads(line.strip()))
    
    df_stats = pd.DataFrame(sample_chunks)
    doc_counts = df_stats.groupby("doc_id")["total_chunks"].first()
    
    stats_report = {
        "processed_date": datetime.now().isoformat(),
        "data_type": "验证样本（1000篇）",
        "input_file": str(SAMPLE_OUTPUT),
        "processed_docs": len(doc_counts),
        "total_chunks": len(df_stats),
        "single_count": int((doc_counts == 1).sum()),
        "multi_count": int((doc_counts > 1).sum()),
        "chunk_size": config["retrieval_chunk_size"],
        "chunk_overlap": config["retrieval_chunk_overlap"],
        "token_limit": config["retrieval_token_limit"],
    }
    
    # 保存统计（使用 _sample 后缀区分）
    stats_path = Path("../outputs/samples/chunking_stats_sample.json")
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats_report, f, indent=2, ensure_ascii=False)
    
    print(f"验证样本统计已保存: {stats_path.resolve()}")
    print(json.dumps(stats_report, indent=2, ensure_ascii=False))
else:
    print(f"验证样本文件不存在: {SAMPLE_OUTPUT}")

验证样本统计已保存: D:\谷歌\03 文档解析与分割\outputs\tables\chunking_stats_sample.json
{
  "processed_date": "2026-05-27T13:06:42.280890",
  "data_type": "验证样本（1000篇）",
  "input_file": "..\\data\\processed\\chunks_sample.jsonl",
  "processed_docs": 1000,
  "total_chunks": 1267,
  "single_count": 888,
  "multi_count": 112,
  "chunk_size": 400,
  "chunk_overlap": 80,
  "token_limit": 512
}


---
## 【C4】预览结果

查看输出的 chunk 数据格式和样本。

In [6]:
# 预览验证样本文件（从本地副本读取）
if SAMPLE_OUTPUT.exists():
    chunks_preview = []
    with open(SAMPLE_OUTPUT, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 20:
                break
            chunks_preview.append(json.loads(line.strip()))
    
    df_chunks = pd.DataFrame(chunks_preview)
    print(f"数据来源: {SAMPLE_OUTPUT}")
    print(f"输出字段: {list(df_chunks.columns)}")
    print()
    display(df_chunks[["chunk_id", "doc_id", "chunk_index", "total_chunks", "token_count", "strategy"]])
else:
    print(f"验证样本文件不存在: {SAMPLE_OUTPUT}")

数据来源: ..\data\processed\chunks_sample.jsonl
输出字段: ['chunk_id', 'text', 'doc_id', 'chunk_index', 'total_chunks', 'source_title', 'token_count', 'strategy']



,chunk_id,doc_id,chunk_index,total_chunks,token_count,strategy
0,PMC176545,PMC176545,0,1,480,single
1,PMC176546,PMC176546,0,1,297,single
2,PMC193604_chunk0,PMC193604,0,4,11,sliding_window
3,PMC193604_chunk1,PMC193604,1,4,389,sliding_window
4,PMC193604_chunk2,PMC193604,2,4,157,sliding_window
5,PMC193604_chunk3,PMC193604,3,4,36,sliding_window
6,PMC193605,PMC193605,0,1,418,single
7,PMC212319,PMC212319,0,1,361,single
8,PMC212687,PMC212687,0,1,447,single
9,PMC212689,PMC212689,0,1,27,single


In [7]:
# 查看多块分割的样例（从验证样本）
if SAMPLE_OUTPUT.exists():
    multi_chunks = [c for c in chunks_preview if c["total_chunks"] > 1]
    if multi_chunks:
        print(f"=== 多块分割样例（共 {len(multi_chunks)} 个 chunk）===")
        for chunk in multi_chunks[:5]:
            print(f"\n[{chunk['chunk_id']}] chunk {chunk['chunk_index']+1}/{chunk['total_chunks']}")
            print(f"  tokens: {chunk['token_count']}")
            print(f"  text: {chunk['text'][:200]}...")
    else:
        print("预览范围内无多块分割样例")

=== 多块分割样例（共 4 个 chunk）===

[PMC193604_chunk0] chunk 1/4
  tokens: 11
  text: Drosophila Free-Running Rhythms Require Intercellular Communication...

[PMC193604_chunk1] chunk 2/4
  tokens: 389
  text: Robust self-sustained oscillations are a ubiquitous characteristic of circadian rhythms. These include Drosophila locomotor activity rhythms, which persist for weeks in constant darkness (DD). Yet the...

[PMC193604_chunk2] chunk 3/4
  tokens: 157
  text: . We showed that the brain circadian clock in Drosophila is clearly distinguishable from the eyes and other rapidly damping peripheral tissues, as it sustains robust molecular oscillations in DD. At t...

[PMC193604_chunk3] chunk 4/4
  tokens: 36
  text: Circadian rhythms are characterized by robust molecular oscillations, which are shown here to require a brain region-specific neuropeptide, PDF, for maintenance and coordination...


---
## 【C5】质量验证

按任务书要求进行质量抽样检查：
1. 总块数，是否超过模型限制
2. 文本质量
3. 是否包含标题
4. 文本是否被不完整截断
5. 多块分割重叠部分检查

In [8]:
# 抽样验证配置
SAMPLE_SIZE = 1000  # 抽样数量
TOKEN_LIMIT = 512   # 模型上限

In [9]:
import random

# 从验证样本进行质量验证
if SAMPLE_OUTPUT.exists():
    # 加载全部验证样本
    all_chunks = []
    with open(SAMPLE_OUTPUT, "r", encoding="utf-8") as f:
        for line in f:
            all_chunks.append(json.loads(line.strip()))
    
    sample = random.sample(all_chunks, min(SAMPLE_SIZE, len(all_chunks)))
    df_sample = pd.DataFrame(sample)
    
    print(f"=== 质量验证（验证样本，抽样 {len(sample)} 个 chunk）===")
    print(f"数据来源: {SAMPLE_OUTPUT}")
    print()
    
    # 1. Token 超限检查
    over_limit = df_sample[df_sample["token_count"] > TOKEN_LIMIT]
    print(f"1. Token 超限（>{TOKEN_LIMIT}）: {len(over_limit)} 个 ({len(over_limit)/len(sample)*100:.2f}%)")
    if len(over_limit) > 0:
        print(f"   最大 token: {df_sample['token_count'].max()}")
    
    # 2. 空文本检查
    empty_text = df_sample[df_sample["text"].str.strip() == ""]
    print(f"2. 空文本: {len(empty_text)} 个")
    
    # 3. 极短文本检查（<50 字符）
    short_text = df_sample[df_sample["text"].str.len() < 50]
    print(f"3. 极短文本（<50字符）: {len(short_text)} 个")
    
    # 4. Token 分布
    print(f"\n4. Token 分布:")
    print(f"   mean: {df_sample['token_count'].mean():.1f}")
    print(f"   median: {df_sample['token_count'].median():.1f}")
    print(f"   P95: {df_sample['token_count'].quantile(0.95):.1f}")
    print(f"   max: {df_sample['token_count'].max()}")
    
    # 5. 策略分布
    print(f"\n5. 策略分布:")
    print(df_sample["strategy"].value_counts())
else:
    print(f"验证样本文件不存在: {SAMPLE_OUTPUT}")

=== 质量验证（验证样本，抽样 1000 个 chunk）===
数据来源: ..\data\processed\chunks_sample.jsonl

1. Token 超限（>512）: 0 个 (0.00%)
2. 空文本: 0 个
3. 极短文本（<50字符）: 3 个

4. Token 分布:
   mean: 262.0
   median: 292.5
   P95: 472.0
   max: 512

5. 策略分布:
strategy
single            699
sliding_window    301
Name: count, dtype: int64


In [10]:
# 多块分割重叠检查（从验证样本）
if SAMPLE_OUTPUT.exists() and 'df_sample' in dir():
    multi_doc_ids = df_sample[df_sample["total_chunks"] > 1]["doc_id"].unique()
    
    if len(multi_doc_ids) > 0:
        # 取一个多块文档检查重叠
        sample_doc_id = multi_doc_ids[0]
        doc_chunks = [c for c in all_chunks if c["doc_id"] == sample_doc_id]
        doc_chunks.sort(key=lambda x: x["chunk_index"])
        
        print(f"=== 多块重叠检查样例: {sample_doc_id} ===")
        print(f"总块数: {len(doc_chunks)}")
        print()
        
        for i, chunk in enumerate(doc_chunks):
            print(f"--- Chunk {i} (tokens: {chunk['token_count']}) ---")
            print(chunk["text"][:300])
            print("..." if len(chunk["text"]) > 300 else "")
            print()
    else:
        print("抽样中无多块文档")

=== 多块重叠检查样例: PMC516440 ===
总块数: 3

--- Chunk 0 (tokens: 42) ---
Flagellin acting via TLR5 is the major activator of key signaling pathways leading to NF-κB and proinflammatory gene program activation in intestinal epithelial cells


--- Chunk 1 (tokens: 356) ---
Infection of intestinal epithelial cells by pathogenic Salmonella leads to activation of signaling cascades that ultimately initiate the proinflammatory gene program. The transcription factor NF-κB is a key regulator/activator of this gene program and is potently activated. We explored the mechanism
...

--- Chunk 2 (tokens: 151) ---
These collective results provide the evidence that flagellin acts as the main determinant of Salmonella mediated NF-κB and proinflammatory signaling and gene activation by this flagellated pathogen. In addition, expression of the fli C gene appears to play an important role in the proper functioning
...



---
## 【C6】导出质量报告

In [11]:
# 生成验证样本的质量报告（使用 _sample 后缀区分）
if SAMPLE_OUTPUT.exists() and 'df_sample' in dir():
    quality_report = {
        "验证日期": datetime.now().isoformat(),
        "数据类型": "验证样本（1000篇）",
        "数据来源": str(SAMPLE_OUTPUT),
        "抽样数量": len(df_sample),
        "token超限数": int(len(over_limit)),
        "空文本数": int(len(empty_text)),
        "极短文本数": int(len(short_text)),
        "token_mean": round(df_sample['token_count'].mean(), 2),
        "token_median": round(df_sample['token_count'].median(), 2),
        "token_p95": round(df_sample['token_count'].quantile(0.95), 2),
        "token_max": int(df_sample['token_count'].max()),
        "single_ratio": round(len(df_sample[df_sample['strategy']=='single']) / len(df_sample), 4),
        "multi_ratio": round(len(df_sample[df_sample['strategy']=='sliding_window']) / len(df_sample), 4),
    }
    
    # 使用 _sample 后缀区分全量报告
    report_path = Path("../outputs/tables/quality_report_sample.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(quality_report, f, indent=2, ensure_ascii=False)
    
    print(f"验证样本质量报告已保存: {report_path.resolve()}")
    print(json.dumps(quality_report, indent=2, ensure_ascii=False))

验证样本质量报告已保存: D:\谷歌\03 文档解析与分割\outputs\tables\quality_report_sample.json
{
  "验证日期": "2026-05-27T13:07:06.894816",
  "数据类型": "验证样本（1000篇）",
  "数据来源": "..\\data\\processed\\chunks_sample.jsonl",
  "抽样数量": 1000,
  "token超限数": 0,
  "空文本数": 0,
  "极短文本数": 3,
  "token_mean": 261.96,
  "token_median": 292.5,
  "token_p95": 472.0,
  "token_max": 512,
  "single_ratio": 0.699,
  "multi_ratio": 0.301
}


---
## 完成

### 交付产物（验证样本）

| 产物 | 路径 | Git |
|------|------|-----|
| 验证样本数据（1000篇） | `data/processed/chunks_sample.jsonl` | ✅ 可提交 |
| 验证样本统计 | `outputs/tables/chunking_stats_sample.json` | ✅ 可提交 |
| 验证样本质量报告 | `outputs/tables/quality_report_sample.json` | ✅ 可提交 |

> **注意**：本 notebook 用于验证样本分析，从本地 `chunks_sample.jsonl` 读取数据。
> 全量处理请使用 `doc-chunking-full.ipynb`。

为了区分数据，本次生成的样本内容移入 \03 文档解析与分割\outputs\samples